---

In [2]:
import stim
from fractions import Fraction
from pyzx import Circuit, GraphState, draw_d3

boh = stim.Tableau.from_stabilizers([
    stim.PauliString("XZZ"),
    stim.PauliString("ZXZ"),
    stim.PauliString("ZZX"),
], allow_underconstrained=True)

circ = boh.to_circuit()
qasm = circ.to_qasm(open_qasm_version=3)
pyzx_circ = Circuit.from_qasm(qasm)
g_boh = GraphState.from_circuit(pyzx_circ, 0)
g_boh.to_canonical_form(quiet=True)
draw_d3(g_boh)

spider_simp: 2. 2. 1.  3 iterations
id_simp: 1.  1 iterations
spider_simp: 2. 1.  2 iterations
()


In [ ]:

qasm = stim.Tableau.from_stabilizers([stim.PauliString("XX"),stim.PauliString("ZZ"),]).to_circuit().to_qasm(open_qasm_version=3)
qasm2 = stim.Tableau.from_stabilizers([stim.PauliString("IIIXXXX"),stim.PauliString("IXXIIXX"),stim.PauliString("XIXIXIX"), stim.PauliString("IIIZZZZ"), stim.PauliString("IZZIIZZ"), stim.PauliString("ZIZIZIZ")], allow_underconstrained=True).to_circuit().to_qasm(open_qasm_version=3)
qasm_5qubit = stim.Tableau.from_stabilizers([
    stim.PauliString("XZZXI"),
    stim.PauliString("IXZZX"), 
    stim.PauliString("XIXZZ"),
    stim.PauliString("ZXIXZ")
], allow_underconstrained=True).to_circuit().to_qasm(open_qasm_version=3)
shor_code = stim.Tableau.from_stabilizers([
    stim.PauliString("ZZIIIIIII"),
    stim.PauliString("ZIZIIIIII"),
    stim.PauliString("IIIZZIIII"),
    stim.PauliString("IIIZIZIII"),
    stim.PauliString("IIIIIIZZI"),
    stim.PauliString("IIIIIIZIZ"),
    stim.PauliString("XXXXXXIII"),
    stim.PauliString("IIIXXXXXX"),
], allow_underconstrained=True).to_circuit().to_qasm(open_qasm_version=3)
qasm_7qubit = stim.Tableau.from_stabilizers([
    stim.PauliString("IIIXXXX"),
    stim.PauliString("IXXIIXX"),
    stim.PauliString("XIXIXIX"),
    stim.PauliString("IIIZZZZ"),
    stim.PauliString("IZZIIZZ"),
    stim.PauliString("ZIZIZIZ")
], allow_underconstrained=True).to_circuit(method="elimination").to_qasm(open_qasm_version=3) # O(n^2)
# shor_code.diagram()


# s = stim.TableauSimulator()
# s.do_circuit(shor_code)
# s.canonical_stabilizers()
n_qasm = 7
k = 1
print(qasm_7qubit)



OPENQASM 3.0;
include "stdgates.inc";

qreg q[7];

cx q[5], q[0];
cx q[0], q[5];
cx q[5], q[0];
h q[2];
cx q[2], q[0];
cx q[3], q[0];
cx q[4], q[0];
cx q[6], q[0];
h q[1];
cx q[1], q[2];
cx q[1], q[4];
cx q[2], q[4];
cx q[2], q[6];
cx q[5], q[3];
cx q[3], q[5];
cx q[5], q[3];
h q[3];
cx q[3], q[5];
cx q[3], q[6];
cx q[5], q[4];
cx q[4], q[5];
cx q[5], q[4];
cx q[5], q[4];
cx q[6], q[5];



In [1]:
# from ..pyzx.graph_states import GraphState
from pyzx import Circuit
from pyzx import *
from pyzx import to_graph_like, clifford_simp
from pyzx import draw_d3
from pyzx import GraphState
import stim
import re

def stim_qasm3_to_pyzx_qasm2(qasm: str, decompose_cz: bool = False) -> str:
    q = qasm
    # q = re.sub(r'OPENQASM 3\.0;', 'OPENQASM 2.0;', q)
    # q = re.sub(r'include "stdgates\.inc";\n', 'include "qelib1.inc";\n', q)
    # remove rx definition
    q = re.sub(r'def\s+rx\(qubit q0\)\s*\{[^}]*\}\n+', '', q)
    # replace rx(q[i]); with h q[i];
    q = re.sub(r'rx\s*\(\s*q\[(\d+)\]\s*\)\s*;', r'h q[\1];', q)
    # remove reset if still present
    q = re.sub(r'reset\s+q\[(\d+)\];', '', q)
    # remove barriers
    # q = re.sub(r'barrier q;\n', '', q)
    # compress blank lines
    # q = re.sub(r'\n\s*\n+', '\n', q).strip() + '\n'
    # if decompose_cz:
    #     lines = []
    #     for line in q.splitlines():
    #         m = re.match(r'cz q\[(\d+)\],\s*q\[(\d+)\];', line.strip())
    #         if m:
    #             a, b = m.groups()
    #             lines.append(f'h q[{b}];')
    #             lines.append(f'cx q[{a}], q[{b}];')
    #             lines.append(f'h q[{b}];')
    #         else:
    #             lines.append(line)
    #     q = '\n'.join(lines) + '\n'
    return q

def tableau_to_graph(list, quiet = True):
    n = len(list[0])
    k = n - len(list)
    stabilizers = [stim.PauliString(x) for x in list]
    tableau = stim.Tableau.from_stabilizers(stabilizers, allow_underconstrained=True)
    stim_circ = tableau.to_circuit(method="elimination")
    print(stim_circ)
    qasm = stim_circ.to_qasm(open_qasm_version=3)
    pyzx_circ = Circuit.from_qasm(qasm)

    g = pyzx_circ.to_graph()


    input_state = "0"*(n-k) + "/"*k
    g.apply_state(input_state)

    # Put |0> as input for n-k qubits
    # inputs = pyzx_circ.inputs()
    # for i in range(len(pyzx_circ.inputs())-k):
    #     q = pyzx_circ.inputs()[i]

    #     # Using |0> as input (which is |0> = H|+> )
    #     pyzx_circ.set_type(q, VertexType.Z)
    #     neigh = [x for x in pyzx_circ.neighbors(q)]
    #     neigh = neigh[0]
    #     e = pyzx_circ.edge(q, neigh)
    #     if pyzx_circ.edge_type(e) == EdgeType.HADAMARD:
    #         pyzx_circ.set_edge_type(e, EdgeType.SIMPLE)
    #     else:
    #         pyzx_circ.set_edge_type(e, EdgeType.HADAMARD)

    g = GraphState.from_clifford_diagram(g)
    draw_d3(g)
    g.to_canonical_form(quiet = quiet)
    return g
    
tableau_to_graph(["IZZ", "ZIZ"])

CX 1 0 0 1 1 0 2 0 2 1
spider_simp: 2. 1.  2 iterations
id_simp: 1.  1 iterations
spider_simp: 1.  1 iterations
pivot_simp: 1. 1. 1. 1.  4 iterations
id_simp: 2.  1 iterations


(2,)


In [ ]:
import networkx as nx
from pyvis.network import Network
import streamlit as st

def pyzx_graph_to_pyvis(g):
    nxg = nx.Graph()
    for v in g.get_states():
        v_type = g.type(v)
        if g.get_bound(v) in g.inputs():
            color = "green"
        else:
            color = "blue"
        if v_type != VertexType.BOUNDARY:
            nxg.add_node(v, 
                        label=str(v), 
                        color=color, 
                        title=f"Type: {v_type.name}")
            
    for e in g.edges():
        edge_type = g.edge_type(g.edge(*e))
        edge_color = "red" if edge_type == EdgeType.HADAMARD else "black"
        if g.type(e[0]) != VertexType.BOUNDARY and g.type(e[1]) != VertexType.BOUNDARY:
            nxg.add_edge(e[0], e[1])
            nxg[e[0]][e[1]]['color'] = edge_color

    

    # # Create PyVis network
    net = Network(notebook=False, directed=False)
    net.from_nx(nxg)
    net.toggle_physics(True)
    return net

    # net.show_buttons(filter_=['physics'])

    # html_path = "pyzx_graph.html"
    # net.save_graph(html_path)  # ✅ does not require render()

pyzx_graph_to_pyvis(g)


AttributeError: 'Graph' object has no attribute 'print'